In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# ==========================================
# 1. SETUP BASE VARIABLES
# ==========================================
start_date = datetime(2024, 1, 1)
base_vendors = [
    "ABC Corp", "QuickFix IT", "Global Supplies", "TechNova", 
    "Summit Group", "Apex Holdings", "BlueWave Media", "Zenith Logistics"
]

# The "Ghost Vendor" Trap (Similarities designed to hit the 75-95% threshold)
ghost_vendors = [
    "A.B.C. Corp", "ABC Corporation", "ABC Corp.",           # Matches ABC Corp
    "QuickFix I.T.", "Quick Fix IT", "QuickFix IT Solutions",# Matches QuickFix IT
    "Global Supply Co.", "Global Supplies Inc",              # Matches Global Supplies
    "Tech Nova Ltd", "TechNova L.T.D.", "TechNova Inc",      # Matches TechNova
    "Apex Holding", "Apex Holdings LLC"                      # Matches Apex Holdings
]

data = []

# ==========================================
# 2. GENERATE NORMAL BEHAVIOR (~809 rows)
# ==========================================
# We generate a large baseline that naturally follows Benford's Law
# plus enough rows to reach exactly 1,000 total records.
TOTAL_ROWS = 1000
GHOST_ROWS = 80
PADDING_ROWS = 100
INTEGRITY_ROWS = 9
OUTLIER_ROWS = 2
BASELINE_ROWS = TOTAL_ROWS - (GHOST_ROWS + PADDING_ROWS + INTEGRITY_ROWS + OUTLIER_ROWS)

print("Generating baseline transactions...")
for i in range(BASELINE_ROWS):
    tx_id = f"TXN{10000 + i}"
    # Random date within 90 days
    date = (start_date + timedelta(days=random.randint(0, 90))).strftime("%Y-%m-%d")
    
    # Generate amounts that roughly follow Benford naturally (1s, 2s, 3s are common)
    leading_digit = np.random.choice([1, 2, 3, 4, 5, 6, 7], p=[0.30, 0.20, 0.15, 0.10, 0.10, 0.10, 0.05])
    amount = float(f"{leading_digit}{random.randint(10, 999)}.{random.randint(0, 99):02d}")
    
    source = random.choice(["Finance Dept", "HR Dept", "Admin Dept", "IT Dept", "Operations"])
    dest = random.choice(base_vendors)
    
    data.append([tx_id, date, amount, source, dest])

# ==========================================
# 3. INJECT THE HACKATHON TRAPS (~200 rows)
# ==========================================
print("Injecting Ghost Vendors, Invoice Padding, and Data Corruption...")

# TRAP A: Ghost Vendors (Fuzzy Matching)
# Injecting 80 transactions to slightly altered vendor names
for i in range(80):
    tx_id = f"TXN{20000 + i}"
    date = (start_date + timedelta(days=random.randint(0, 90))).strftime("%Y-%m-%d")
    amount = round(random.uniform(2000, 5000), 2)
    data.append([tx_id, date, amount, "Finance Dept", random.choice(ghost_vendors)])

# TRAP B: Benford's Law Violation (Invoice Padding)
# Injecting 100 transactions starting with '8' and '9' to brutally break the Benford curve
for i in range(100):
    tx_id = f"TXN{30000 + i}"
    date = (start_date + timedelta(days=random.randint(0, 90))).strftime("%Y-%m-%d")
    amount = float(f"{random.choice([8, 9])}{random.randint(100, 999)}.{random.randint(0, 99):02d}")
    data.append([tx_id, date, amount, "Admin Dept", "Summit Group"])

# TRAP C: Data Integrity Destruction (Readiness Score Drop)
# These will force your Readiness Score to drop and list specific validation errors
data.append(["TXN40001", "2024-02-14", np.nan, "HR Dept", "TechNova"])                 # Missing Amount
data.append(["TXN40002", "2024-02-15", 500.00, "IT Dept", np.nan])                     # Missing Vendor
data.append(["TXN40003", "2024-02-15", 750.00, np.nan, "ABC Corp"])                    # Missing Source
data.append(["TXN10005", "2024-02-16", 1200.00, "Finance Dept", "ABC Corp"])           # Duplicate ID (Matches row 5)
data.append(["TXN10005", "2024-02-16", 1200.00, "Finance Dept", "ABC Corp"])           # Another Duplicate ID
data.append(["TXN40004", "2024-02-17", "Five Thousand", "Admin Dept", "QuickFix IT"])  # String in Float column
data.append(["TXN40005", "2024-02-18", "1,200.50", "Admin Dept", "QuickFix IT"])       # Comma in Float column
data.append(["TXN40006", "Not-A-Date", 800.00, "HR Dept", "Global Supplies"])          # Bad Date format
data.append(["TXN40007", "15-14-2024", 900.00, "HR Dept", "Global Supplies"])          # Impossible Date

# TRAP D: Anomaly Detection (Isolation Forest & Network Loops)
# Massive outliers that your algorithm will flag as 99% Risk
data.append(["TXN50001", "2024-01-07", 995000.00, "Finance Dept", "TechNova L.T.D."])  # Massive amount to ghost vendor
data.append(["TXN50002", "2024-01-14", 850000.00, "Operations", "Apex Holding"])       # Another massive outlier

# ==========================================
# 4. FORMAT, SHUFFLE, AND EXPORT
# ==========================================
df = pd.DataFrame(data, columns=["transaction_id", "timestamp", "amount", "source_entity", "destination_entity"])
assert len(df) == TOTAL_ROWS, f"Expected {TOTAL_ROWS} rows but generated {len(df)}"

# Shuffle the rows so the traps are hidden randomly throughout the 1,000 rows
output_filename = "spit_demo_ledger_1000.csv"
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.to_csv(output_filename, index=False)

print(f"✅ Success! {len(df)} rows generated and saved to '{output_filename}'.")
print("🔥 The traps are set. Upload this 1,000-row monster to your dashboard!")

C:\Users\MadhavIsrani\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Generating baseline transactions...
Injecting Ghost Vendors, Invoice Padding, and Data Corruption...
✅ Success! 991 rows generated and saved to 'spit_demo_ledger_1000.csv'.
🔥 The traps are set. Upload this 1,000-row monster to your dashboard!
